# Aula 9 — Graph RAG com LightRAG, OpenSearch e vLLM
## MBA: RAG & CAG Aplicados a Direito e Segurança Pública

**Carga horária:** 5 horas | **Proporção:** 30% teoria / 70% prática

---

> **ABNT NBR 6023:2018** — Este material segue as normas da Associação Brasileira de Normas Técnicas para produção acadêmica.

---

## Objetivos de Aprendizagem

Ao concluir esta aula, o aluno será capaz de:

1. Compreender a arquitetura do Graph RAG e quando ele supera o RAG textual convencional;
2. Configurar o LightRAG com armazenamento vetorial no OpenSearch e LLM via vLLM;
3. Realizar queries nos modos **local**, **global**, **hybrid** e **naive** sobre corpus jurídicos;
4. Integrar embeddings locais (BAAI/bge-m3) para privacidade e controle de dados sensíveis;
5. Construir pipelines de investigação criminal usando grafos de conhecimento.

---

## Sumário

1. [Fundamentos Teóricos do Graph RAG](#1-fundamentos-teóricos)
2. [Arquitetura do LightRAG](#2-arquitetura-lightrag)
3. [Limitações do RAG Textual e Motivação para Grafos](#3-limitações)
4. [Modos de Query: Local, Global, Hybrid e Naive](#4-modos-query)
5. [Stack Tecnológico: vLLM + OpenSearch + Embeddings Locais](#5-stack)
6. [Aplicações em Direito e Segurança Pública](#6-aplicações)
7. [Referências](#7-referências)

---
## 1. Fundamentos Teóricos do Graph RAG

### 1.1 O Problema que o Graph RAG Resolve

O RAG convencional (Aulas 2–6) opera sobre **chunks de texto independentes**: cada trecho é vetorizado, indexado e recuperado individualmente. Esse modelo funciona bem para perguntas factuais diretas, mas falha em cenários que exigem **raciocínio multi-hop** — quando a resposta depende de conectar múltiplas entidades através de relações.

**Exemplo prático no contexto jurídico:**

```
Pergunta: "Quais as conexões entre o réu Silva e organizações criminosas 
          investigadas pelo MPF nos últimos dois anos?"
```

O RAG textual retornaria chunks sobre Silva e chunks sobre o MPF separadamente, sem **inferir as conexões** entre eles. O Graph RAG, ao contrário, navega pelo grafo:

```
Silva → investigado_por → MPF
Silva → associado_a → Organização_Alpha
Organização_Alpha → opera_em → SP, RJ, MG
MPF → conduz → Operação_Integração
```

### 1.2 Grafos de Conhecimento: Conceitos Essenciais

Um **grafo de conhecimento** (Knowledge Graph — KG) é uma estrutura de dados que representa entidades do mundo real e suas relações em formato de triplas:

```
(Sujeito) → [Relação] → (Objeto)
(Lei_12850) → [define] → (Organização_Criminosa)
(STF) → [julgou] → (HC_127483)
(HC_127483) → [trata_de] → (Colaboração_Premiada)
```

**Componentes estruturais:**

- **Nós (Entidades):** pessoas, leis, tribunais, operações, crimes;
- **Arestas (Relações):** verbos que conectam entidades;
- **Comunidades:** agrupamentos de nós densamente conectados;
- **Atributos:** propriedades dos nós (data, jurisdição, valor).

### 1.3 Como o LLM Extrai o Grafo Automaticamente

O LightRAG (Guo et al., 2024) automatiza a construção do grafo de conhecimento com o seguinte pipeline:

```
Documento Bruto
      │
      ▼
  Chunking (por tokens)
      │
      ▼
  LLM: Extração de Entidades + Relações
      │
      ├── Entidades → índice vetorial (OpenSearch kNN)
      └── Relações → grafo de adjacência
      │
      ▼
  Deduplicação e Mesclagem de Entidades
      │
      ▼
  Detecção de Comunidades (algoritmo de Leiden)
      │
      ▼
  Geração de Sumários por Comunidade
      │
      ▼
  Grafo de Conhecimento Indexado
```

O prompt de extração usado pelo LightRAG instrui o LLM a identificar entidades nomeadas e seus relacionamentos em linguagem natural, convertendo-os em triplas estruturadas.

---
## 2. Arquitetura do LightRAG

### 2.1 Visão Geral

O LightRAG (HKUDS/LightRAG, 2024) implementa o paradigma **Graph RAG** com foco em eficiência e modularidade. Sua arquitetura suporta múltiplos backends de armazenamento e LLMs via API compatível com OpenAI.

```
┌─────────────────────────────────────────────────────┐
│                    LightRAG Core                    │
├──────────────┬──────────────┬───────────────────────┤
│  LLM Backend │  Embedding   │   Storage Backend     │
│   (vLLM)     │  (BAAI/bge)  │   (OpenSearch)        │
├──────────────┴──────────────┴───────────────────────┤
│  Indexação: chunking → extração → grafo → vetores   │
│  Query: local | global | hybrid | naive             │
└─────────────────────────────────────────────────────┘
```

### 2.2 Tipos de Armazenamento no LightRAG

O LightRAG mantém **quatro tipos de índice** internamente:

| Tipo | Conteúdo | Backend (OpenSearch) |
|------|----------|----------------------|
| **KV Store (Docs)** | Chunks originais dos documentos | Índice `lightrag-docs` |
| **KV Store (LLM Cache)** | Cache de chamadas ao LLM | Índice `lightrag-llm-cache` |
| **Vector Store** | Embeddings de entidades | Índice `lightrag-entities` (kNN) |
| **Graph Store** | Grafo de entidades e relações | Índice `lightrag-graph` |

### 2.3 Integração OpenSearch como Backend Unificado

Desde março de 2026, o LightRAG suporta OpenSearch como backend unificado para todos os quatro tipos de armazenamento. Isso é especialmente relevante para ambientes de segurança pública, onde:

- **Dados permanecem on-premise** (sem envio para APIs externas);
- **Controle de acesso** via IAM/RBAC do OpenSearch;
- **Auditoria** de todas as queries;
- **Escalabilidade** horizontal para grandes corpora.

### 2.4 vLLM como Backend LLM

O **vLLM** (Virtual Large Language Model) é um servidor de inferência de alto desempenho que:

- Implementa **PagedAttention** para gerenciamento eficiente de memória KV;
- Expõe API compatível com OpenAI (`/v1/chat/completions`);
- Suporta modelos como Llama-3, Mistral, Qwen e outros;
- Permite deployment local em GPUs on-premise.

Para o LightRAG, basta apontar `base_url` para o endpoint vLLM — não são necessárias modificações no código.

### 2.5 Embeddings Locais com BAAI/bge-m3

O modelo `BAAI/bge-m3` (Beijing Academy of AI) é um modelo de embedding multilíngue de última geração que:

- Suporta **português** nativamente;
- Produz vetores de dimensão 1024;
- Roda localmente via `FlagEmbedding` ou `sentence-transformers`;
- É gratuito e open-source (licença MIT).

> **Importante para Segurança Pública:** documentos sigilosos nunca devem ser enviados para APIs externas (OpenAI, Cohere etc.). O uso de embeddings locais garante que os dados permanecem sob controle da instituição.

---
## 3. Limitações do RAG Textual e Motivação para Grafos

### 3.1 Quadro Comparativo

| Critério | RAG Textual | Graph RAG (LightRAG) |
|----------|-------------|----------------------|
| **Raciocínio multi-hop** | ❌ Limitado | ✅ Nativo via grafo |
| **Perguntas globais** | ❌ Sem visão do corpus | ✅ Comunidades temáticas |
| **Relações entre entidades** | ❌ Implícitas | ✅ Explícitas e navegáveis |
| **Custo de indexação** | 🟡 Médio | 🔴 Alto (extração LLM) |
| **Atualização incremental** | ✅ Fácil | ✅ Suportado |
| **Privacidade on-premise** | ✅ Com embeddings locais | ✅ Com vLLM + OpenSearch |

### 3.2 Quando Usar Graph RAG no Contexto Jurídico

**Use Graph RAG quando a pergunta envolve:**

- Mapeamento de redes criminosas e conexões entre investigados;
- Rastreamento de jurisprudência — "quais decisões do STJ citam o HC X?";
- Análise de fluxo financeiro entre entidades (lavagem de dinheiro);
- Identificação de padrões em operações policiais ao longo do tempo;
- Visão temática de um corpus — "quais os principais temas do acervo?".

**Mantenha RAG textual convencional quando:**

- A pergunta é pontual e factual ("Qual a pena do art. 33 da Lei 11.343?");
- O corpus é pequeno e atualizado frequentemente;
- Custo computacional é uma restrição crítica.

---
## 4. Modos de Query: Local, Global, Hybrid e Naive

O LightRAG implementa quatro estratégias de recuperação:

### 4.1 Modo Naive
Recuperação vetorial simples, sem uso do grafo. Equivale ao RAG convencional. Útil como baseline.

```
Query → Embedding → kNN no OpenSearch → Chunks → LLM → Resposta
```

### 4.2 Modo Local
Foca em **entidades específicas** e seus vizinhos imediatos no grafo. Ideal para perguntas sobre pessoas, leis ou casos concretos.

```
Query → Entidades relevantes → Subgrafo local → Contexto → LLM → Resposta
Exemplo: "Quem são os coautores investigados com Silva?"
```

### 4.3 Modo Global
Usa **sumários de comunidades** para responder perguntas sobre temas amplos do corpus. Ideal para visão panorâmica.

```
Query → Comunidades relevantes → Sumários → LLM → Resposta
Exemplo: "Quais os principais temas jurídicos no acervo?"
```

### 4.4 Modo Hybrid (Recomendado)
Combina recuperação local (entidades) com global (comunidades) mais chunks textuais. Melhor qualidade geral.

```
Query → [Subgrafo local] + [Sumários globais] + [kNN chunks] → LLM → Resposta
```

| Modo | Velocidade | Qualidade | Uso Recomendado |
|------|-----------|-----------|------------------|
| naive | ⚡⚡⚡ | ⭐⭐ | Baseline, testes |
| local | ⚡⚡ | ⭐⭐⭐ | Perguntas sobre entidades |
| global | ⚡ | ⭐⭐⭐ | Perguntas temáticas |
| hybrid | ⚡ | ⭐⭐⭐⭐⭐ | Produção geral |

---
## 5. Stack Tecnológico: vLLM + OpenSearch + Embeddings Locais

### 5.1 Diagrama da Arquitetura Completa

```
┌─────────────────────────────────────────────────────────────┐
│                    AMBIENTE ON-PREMISE                      │
│                                                             │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐  │
│  │   Documentos │    │    vLLM      │    │  OpenSearch  │  │
│  │   Jurídicos  │    │  (Llama-3)   │    │  (Cluster)   │  │
│  │   Sigilosos  │    │  :8000/v1    │    │  :9200       │  │
│  └──────┬───────┘    └──────┬───────┘    └──────┬───────┘  │
│         │                  │                   │           │
│         ▼                  ▼                   ▼           │
│  ┌─────────────────────────────────────────────────────┐   │
│  │                   LightRAG Core                     │   │
│  │  ┌─────────────┐  ┌─────────────┐  ┌────────────┐  │   │
│  │  │  Indexação  │  │   Extração  │  │   Query    │  │   │
│  │  │  (Chunking) │  │  (Grafo KG) │  │  Engine    │  │   │
│  │  └─────────────┘  └─────────────┘  └────────────┘  │   │
│  └─────────────────────────────────────────────────────┘   │
│         ▲                                                   │
│  ┌──────┴───────┐                                          │
│  │  BAAI/bge-m3 │  ← Embeddings locais (sem API externa)  │
│  │  (local GPU) │                                          │
│  └──────────────┘                                          │
└─────────────────────────────────────────────────────────────┘
```

### 5.2 Requisitos de Hardware

| Componente | Mínimo | Recomendado |
|-----------|--------|-------------|
| GPU para vLLM | 24GB VRAM (modelo 7B) | 80GB VRAM (modelo 70B) |
| GPU para embeddings | 4GB VRAM | 8GB VRAM |
| OpenSearch RAM | 16GB | 64GB |
| Armazenamento | 100GB SSD | 1TB NVMe |

> **Nota para o Google Colab:** nos labs práticos, usaremos simulações com modelos menores e OpenSearch em Docker. Para produção em segurança pública, o deployment deve ser on-premise.

---
## 6. Aplicações em Direito e Segurança Pública

### 6.1 Mapeamento de Redes Criminosas

O Graph RAG é particularmente poderoso para análise de redes criminosas porque naturalmente representa a estrutura hierárquica de organizações: liderança → gerência → operadores, com relações de comando, financiamento e execução.

**Caso de uso:** dado um corpus de relatórios de inteligência, o investigador pode perguntar:
- *"Quem financia a organização identificada na Operação X?"* (query local)
- *"Quais padrões de lavagem de dinheiro aparecem nas operações dos últimos 5 anos?"* (query global)

### 6.2 Análise de Jurisprudência Multi-hop

Em acórdãos e decisões judiciais, o raciocínio jurídico frequentemente envolve múltiplas referências cruzadas: um acórdão cita um precedente, que por sua vez aplica uma súmula, que foi editada com base em outra série de julgamentos.

O Graph RAG permite navegar essa cadeia de citações automaticamente.

### 6.3 Conformidade com LGPD e Sigilo Investigativo

Para dados sensíveis de investigações criminais e dados pessoais (protegidos pela Lei 13.709/2018 — LGPD), o stack on-premise (vLLM + OpenSearch local + embeddings BAAI/bge-m3) garante que:

1. Nenhum dado trafega para servidores externos;
2. O controle de acesso é gerenciado internamente;
3. Auditoria e rastreabilidade são mantidas;
4. Cumprimento das exigências do CONPES e das políticas de segurança da informação dos órgãos de segurança pública.

---
## 7. Referências

EDGE, Darren et al. **From Local to Global: A Graph RAG Approach to Query-Focused Summarization**. arXiv:2404.16130, 2024.

GUO, Zirui et al. **LightRAG: Simple and Fast Retrieval-Augmented Generation**. arXiv:2410.05779, 2024. Disponível em: <https://lightrag.github.io/>. Acesso em: 21 abr. 2026.

HKUDS. **LightRAG: Simple and Fast Retrieval-Augmented Generation**. GitHub, 2024. Disponível em: <https://github.com/HKUDS/LightRAG>. Acesso em: 21 abr. 2026.

KWON, Woosuk et al. **Efficient Memory Management for Large Language Model Serving with PagedAttention**. In: SOSP, 2023.

OPENSEARCH PROJECT. **Hybrid Search**. Documentação Oficial, 2026. Disponível em: <https://docs.opensearch.org/latest/vector-search/ai-search/hybrid-search/>. Acesso em: 21 abr. 2026.

XIAO, Shitao et al. **M3-Embedding: Multi-Linguality, Multi-Functionality, Multi-Granularity Text Embeddings Through Self-Knowledge Distillation**. arXiv:2402.03216, 2024.

BRASIL. **Lei nº 12.850, de 2 de agosto de 2013**. Define organização criminosa. Diário Oficial da União, Brasília, DF, 2013.

BRASIL. **Lei nº 13.709, de 14 de agosto de 2018**. Lei Geral de Proteção de Dados Pessoais. Diário Oficial da União, Brasília, DF, 2018.